# AgentField Tutorial: Production-Ready AI Agent Infrastructure

## What is AgentField?

AgentField is an open-source control plane that treats AI agents as first-class backend services. Unlike traditional agent frameworks designed for chatbots, AgentField provides production infrastructure for autonomous AI systems that make decisions, coordinate across services, and operate at scale.

### Key Features

- **Microservice Architecture**: Each agent runs as an independent API with auto-generated REST endpoints
- **Cryptographic Identity**: Every agent gets a W3C Decentralized Identifier (DID) with Verifiable Credentials for audit trails
- **Async Execution**: Long-running tasks with webhook callbacks, no timeout limits
- **Cross-Agent Communication**: Automatic service discovery and agent-to-agent RPC
- **Shared Memory**: Distributed state management across agent nodes
- **Production Ready**: Health checks, retries, monitoring, and observability built-in

## Installation and Setup

### Prerequisites
- Python 3.10-3.13
- OpenAI, Anthropic, or OpenRouter API key

In [ ]:
# Install AgentField CLI and Python SDK
# !curl -sSf https://agentfield.ai/get | sh

In [3]:
# Install Python dependencies
# !pip install agentfield python-dotenv pydantic requests -q

### Load API Keys from .env File

Create a `.env` file in your project directory with your API keys:

In [1]:
# Create .env file with your API keys
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
AGENTFIELD_SERVER = os.getenv('AGENTFIELD_SERVER', 'http://localhost:8080')

# Verify keys are loaded
print("API keys loaded successfully!")
print(f"AgentField Server: {AGENTFIELD_SERVER}")

API keys loaded successfully!
AgentField Server: http://localhost:8080


### Start the AgentField Control Plane

Run this in a separate terminal:
```bash
af server
```

This starts the control plane on `http://localhost:8080`

![agent-field-1](/products/agentfield/af-1.png)

## Building Your First Agent

### Basic Agent Structure

Every agent consists of:
1. **Agent Configuration**: Node ID, version, AI config
2. **Reasoners**: AI-powered functions with structured outputs
3. **Skills**: Deterministic code for specific tasks

In [18]:
from agentfield import Agent, AIConfig
from pydantic import BaseModel, Field
from typing import List

# Initialize agent with AI configuration
app = Agent(
    node_id="tutorial-agent",
    agentfield_server=AGENTFIELD_SERVER,
    version="1.0.0",
    dev_mode=True,
    ai_config=AIConfig(
        model="openai/gpt-4o",  # LiteLLM format: provider/model
        temperature=0.7,
    ),
)

print(f"Agent '{app.node_id}' initialized successfully!")

🔍 DEBUG: ResultCache initialized with max_size=5000, ttl=120.0
🔍 DEBUG: DID system initialized


Agent 'tutorial-agent' initialized successfully!


### Creating a Simple Skill (No AI)

Skills are deterministic functions that don't require AI.

In [3]:
@app.skill()
async def validate_email(email: str) -> dict:
    """Validate email format"""
    import re
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    is_valid = bool(re.match(pattern, email))
    
    return {
        "email": email,
        "valid": is_valid,
        "message": "Valid email" if is_valid else "Invalid email format"
    }

# Test the skill
result = await validate_email("test@example.com")
print(result)

{'email': 'test@example.com', 'valid': True, 'message': 'Valid email'}


## AI-Powered Reasoners

Reasoners use AI to make intelligent decisions with structured outputs.

In [10]:
# Define structured output schema
class SentimentResult(BaseModel):
    sentiment: str = Field(description="positive, negative, or neutral")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score")
    key_phrases: List[str] = Field(description="Important phrases from text")
    reasoning: str = Field(description="Explanation of sentiment")

@app.reasoner()
async def analyze_sentiment(text: str) -> dict:
    """Analyze sentiment of given text using AI"""
    result = await app.ai(
        system="You are a sentiment analysis expert. Analyze text and provide structured insights.",
        user=f"Analyze the sentiment of this text: {text}",
        schema=SentimentResult
    )
    return result.model_dump()

# Test sentiment analysis
sentiment = await analyze_sentiment("I absolutely love building with AgentField! It's amazing!")
print(f"Sentiment: {sentiment['sentiment']}")
print(f"Confidence: {sentiment['confidence']}")
print(f"Reasoning: {sentiment['reasoning']}")

Sentiment: positive
Confidence: 0.95
Reasoning: The text expresses strong positive feelings indicated by phrases such as 'absolutely love' and 'amazing', which suggest enthusiasm and high satisfaction.


![agent-field-2](/products/agentfield/af-2.png)

In [5]:
# Multi-category classification reasoner
class CategoryResult(BaseModel):
    category: str = Field(description="tech, business, health, or sports")
    confidence: float = Field(ge=0.0, le=1.0)
    subcategory: str = Field(description="Specific subcategory")
    reasoning: str

@app.reasoner()
async def categorize_text(text: str) -> dict:
    """Categorize text into predefined categories"""
    result = await app.ai(
        system="""You are a text categorization expert. 
        Categories: tech, business, health, sports.
        Provide detailed subcategories.""",
        user=f"Categorize this text: {text}",
        schema=CategoryResult
    )
    return result.model_dump()

# Test categorization
category = await categorize_text(
    "Scientists develop new AI algorithm for cancer detection"
)
print(f"Category: {category['category']}")
print(f"Subcategory: {category['subcategory']}")

Category: health
Subcategory: medical technology


## Multi-Step Reasoning

Chain multiple AI calls for complex workflows.

In [11]:
class SearchQueries(BaseModel):
    queries: List[str] = Field(description="List of 3 diverse search queries")

class ResearchSummary(BaseModel):
    topic: str
    key_points: List[str]
    summary: str
    confidence: float

@app.reasoner()
async def research_and_summarize(topic: str) -> dict:
    """Multi-step research workflow"""
    
    # Step 1: Generate search queries
    app.note("Generating search queries...", tags=["progress"])
    queries_result = await app.ai(
        system="Generate 3 diverse search queries for comprehensive research.",
        user=f"Topic: {topic}",
        schema=SearchQueries
    )
    queries = queries_result.queries
    
    # Step 2: Simulate research (in production, call search agent)
    app.note(f"Researching {len(queries)} queries...", tags=["progress"])
    
    # Step 3: Synthesize findings
    app.note("Synthesizing findings...", tags=["progress"])
    summary_result = await app.ai(
        system="Synthesize research findings into a coherent summary with key points.",
        user=f"Topic: {topic}\nQueries explored: {', '.join(queries)}",
        schema=ResearchSummary
    )
    
    return summary_result.model_dump()

# Test multi-step reasoning
research = await research_and_summarize("Quantum Computing Applications")
print(f"Topic: {research['topic']}")
print(f"\nKey Points:")
for point in research['key_points']:
    print(f"  - {point}")
print(f"\nSummary: {research['summary']}")

🔍 DEBUG: NOTE DEBUG: Original api_base: http://localhost:8080/api/v1
🔍 DEBUG: NOTE DEBUG: UI api_base: http://localhost:8080/api/ui/v1
🔍 DEBUG: NOTE DEBUG: Full URL: http://localhost:8080/api/ui/v1/executions/note
🔍 DEBUG: NOTE DEBUG: Payload: {'message': 'Generating search queries...', 'tags': ['progress'], 'timestamp': 1768874503.81846, 'agent_node_id': 'tutorial-agent'}
🔍 DEBUG: NOTE DEBUG: Headers: {'X-Run-ID': 'run_1768873575723_50cc21bc', 'X-Workflow-ID': 'run_1768873575723_50cc21bc', 'X-Parent-Execution-ID': 'exec_1768873575723_333d6aef', 'X-Execution-ID': 'exec_176887450...
🔍 DEBUG: NOTE DEBUG: Response status: 200
🔍 DEBUG: NOTE DEBUG: Response text: {"success":true,"note":{"message":"Generating search queries...","tags":["progress"],"timestamp":"2026-01-20T10:01:43.827919+08:00"},"message":"Note added successfully"}
🔍 DEBUG: ✅ Note successfully sent to http://localhost:8080/api/ui/v1/executions/note
🔍 DEBUG: NOTE DEBUG: Original api_base: http://localhost:8080/api/v1
🔍 DEBUG: 

Topic: Quantum Computing Applications

Key Points:
  - Quantum computing is being explored for its potential to revolutionize financial modeling by optimizing complex calculations and improving risk management strategies.
  - In drug discovery and development, quantum computing can simulate molecular interactions at a faster pace, potentially reducing the time and cost involved in bringing new drugs to market.
  - Quantum computing poses both opportunities and threats to cryptography; while it can potentially crack existing cryptographic codes, it also offers new approaches to encryption that could enhance data security.

Summary: Quantum computing is making significant strides in various fields, with current applications in financial modeling, drug discovery, and cryptography. In finance, it offers enhanced capabilities for complex calculations and risk management. In the pharmaceutical industry, it accelerates the drug development process by simulating molecular interactions. However

## Memory: Shared State Management

Store and retrieve data across agent executions.

In [19]:
@app.skill()
async def save_user_preference(user_id: str, preference: str) -> dict:
    """Store user preference in shared memory"""
    await app.memory.set(
        key=f"user:{user_id}:preference",
        data=preference
    )
    return {"status": "saved", "user_id": user_id, "preference": preference}

@app.skill()
async def get_user_preference(user_id: str) -> dict:
    """Retrieve user preference from memory"""
    preference = await app.memory.get(key=f"user:{user_id}:preference")
    return {"user_id": user_id, "preference": preference}

# Test memory operations
await save_user_preference("user_123", "dark_mode")
stored = await get_user_preference("user_123")
print(f"User preference: {stored['preference']}")

🔍 DEBUG: Memory set operation for key: user:user_123:preference
🔍 DEBUG: Memory set successful for key: user:user_123:preference


User preference: dark_mode


## Cross-Agent Communication

Agents can call other agents' reasoners and skills through the control plane.

In [ ]:
class AnalysisResult(BaseModel):
    sentiment_score: float
    category: str
    combined_insights: str
    confidence: float

@app.reasoner()
async def comprehensive_analysis(text: str) -> dict:
    """Combine multiple agents for comprehensive analysis"""
    
    # Call sentiment analysis reasoner (same agent)
    sentiment = await app.call(
        "tutorial-agent.analyze_sentiment",
        text=text
    )
    
    # Call categorization reasoner (same agent)
    category = await app.call(
        "tutorial-agent.categorize_text",
        text=text
    )
    
    # Synthesize results using AI
    synthesis = await app.ai(
        system="Combine sentiment and category analysis into comprehensive insights.",
        user=f"""Text: {text}
        Sentiment: {sentiment['sentiment']} ({sentiment['confidence']})
        Category: {category['category']}
        Provide combined insights.""",
        schema=AnalysisResult
    )
    
    return synthesis.model_dump()

# Test cross-agent communication
analysis = await comprehensive_analysis(
    "Revolutionary AI breakthrough in medical diagnostics shows 95% accuracy"
)
print(f"Category: {analysis['category']}")
print(f"Sentiment Score: {analysis['sentiment_score']}")
print(f"Insights: {analysis['combined_insights']}")

## Async Execution with Webhooks

For long-running tasks, use async execution with webhook callbacks.

In [ ]:
import requests
from agentfield.types import WebhookConfig

# Example: Queue async execution with webhook
def execute_async_with_webhook(target: str, input_data: dict, webhook_url: str):
    """Execute agent reasoner asynchronously with webhook callback"""
    
    response = requests.post(
        f"{AGENTFIELD_SERVER}/api/v1/execute/async/{target}",
        json={
            "input": input_data,
            "webhook": {
                "url": webhook_url,
                "secret": "your-webhook-secret-key",
                "headers": {
                    "X-Custom-Auth": "Bearer your-token"
                }
            }
        },
        headers={"Content-Type": "application/json"}
    )
    
    return response.json()

# Example webhook handler (FastAPI)
webhook_handler_code = '''
from fastapi import FastAPI, Request

app = FastAPI()

@app.post("/api/agentfield/callback")
async def handle_webhook(request: Request):
    """Handle AgentField webhook callbacks"""
    data = await request.json()
    
    if data["status"] == "completed":
        execution_id = data["execution_id"]
        result = data["result"]
        
        # Process completed execution
        print(f"Execution {execution_id} completed!")
        print(f"Result: {result}")
        
    return {"status": "received"}
'''

print("Async execution with webhooks configured!")
print("\nWebhook handler example:")
print(webhook_handler_code)

## Testing Agent via REST API

Once your agent is running, test it via HTTP requests.

In [ ]:
import requests
import json

def call_agent_api(agent_name: str, reasoner: str, input_data: dict):
    """Call agent reasoner via REST API"""
    url = f"{AGENTFIELD_SERVER}/api/v1/execute/{agent_name}.{reasoner}"
    
    response = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        json={"input": input_data}
    )
    
    return response.json()

# Test sentiment analysis via API
result = call_agent_api(
    agent_name="tutorial-agent",
    reasoner="analyze_sentiment",
    input_data={"text": "AgentField makes building production agents so easy!"}
)

print(json.dumps(result, indent=2))

## Production Deployment Considerations

### Docker Deployment

When running the control plane in Docker:

```bash
# Set callback URL for host-to-container communication
export AGENT_CALLBACK_URL=http://host.docker.internal:8001

# Start control plane
docker run -p 8080:8080 agentfield/control-plane
```

### Environment Variables

Key environment variables:
- `OPENAI_API_KEY`: OpenAI API access
- `ANTHROPIC_API_KEY`: Anthropic Claude access
- `OPENROUTER_API_KEY`: OpenRouter access
- `AGENTFIELD_SERVER`: Control plane URL (default: `http://localhost:8080`)
- `AGENT_CALLBACK_URL`: Agent callback URL for Docker setups

### Kubernetes Deployment

AgentField supports Kubernetes deployment with:
- Horizontal pod autoscaling
- Service mesh integration
- Health checks and readiness probes
- ConfigMaps for agent configuration

## Running Your Agent

### Option 1: Development Mode (Notebook)

Run cells above to test individual components.

### Option 2: Production Mode (CLI)

Create a standalone agent:

```bash
# Initialize new agent
af init my-agent --defaults

# Start control plane (separate terminal)
af server

# Run agent
cd my-agent
python main.py
```

Your agent is now available at:
- Control Plane: `http://localhost:8080`
- Agent Endpoints: `http://localhost:8001`

## Best Practices

### 1. Use Structured Outputs
Always define Pydantic schemas for reasoner outputs to ensure type safety and validation.

### 2. Implement Progress Tracking
Use `app.note()` to stream progress updates for long-running tasks.

### 3. Handle Errors Gracefully
Implement try-catch blocks and return structured error responses.

### 4. Use Memory Wisely
Set appropriate TTL values for cached data to prevent memory bloat.

### 5. Monitor Performance
AgentField provides built-in metrics at `/api/v1/metrics`.

### 6. Security
- Use environment variables for API keys
- Enable webhook signature validation in production
- Leverage DIDs and Verifiable Credentials for audit trails

## Key Differences from Other Frameworks

| Feature | AgentField | LangChain/CrewAI | Traditional APIs |
|---------|------------|------------------|------------------|
| Architecture | Microservices | Monolithic | N/A |
| Service Discovery | Automatic | Manual | N/A |
| Identity | Cryptographic DIDs | None | API Keys |
| Audit Trail | Verifiable Credentials | Logs | Logs |
| Long-Running Tasks | Native support | Limited | Timeouts |
| Multi-Agent | Built-in | Manual | Manual |
| Production Ready | Yes | No | N/A |

## Next Steps

1. **Explore Documentation**: Visit [agentfield.ai/docs](https://agentfield.ai/docs)
2. **Join Community**: GitHub discussions at [github.com/Agent-Field/agentfield](https://github.com/Agent-Field/agentfield)
3. **Build Multi-Agent Systems**: Create specialized agents that coordinate automatically
4. **Deploy to Production**: Follow Kubernetes deployment guide
5. **Implement Observability**: Set up monitoring and alerting

## Resources

- Official Docs: https://agentfield.ai/docs
- GitHub Repository: https://github.com/Agent-Field/agentfield
- Quick Start Guide: https://agentfield.ai/docs/quick-start
- Python SDK: https://pypi.org/project/agentfield/
- Community: GitHub Discussions